In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DUMMY SUBMISSION

In [ ]:
import pandas as pd
sample_sub = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")
sample_sub["label"] = 0
sample_sub.to_csv("submission.csv", index=False)
print(sample_sub.head())
print("Shape:", sample_sub.shape)

# QUESTION UNDERSTANDING
IT IS A COMMENT CATEGORY PREDICTION CHALLENGE-(MULTICLASS TEXT CLASSIFICATION) 

**INPUT DETAILS:**
- Text comment
- Metadata features (upvotes, emoticons, identity attributes, etc.)
- Target label (0,1,2,3)

**OUTPUT DETAILS:**

-Predict the correct category label for unseen comments.
EVALUTION METRIC IS MACRO F1(BECAUSE OF CLASS IMBALANCE)


# IMPORTING LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_sample_weight
import pickle


# LOADING DATA

In [ ]:
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

# EXPLORATARY DATA ANALYSIS

In [ ]:
print(train.info()) #DATASET INFO
print(train.describe()) #STATISTICS
print(train['label'].value_counts()) #LABEL COUNTS
print(train['label'].value_counts(normalize=True) * 100) #LABEL COUNTS PERCENTAGE
#PLOTTING OF DATA TO CHECK IMBALANCE
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=train, hue='label', palette='Set2', legend=False)
plt.title("Label Distribution (Class Imbalance Check)")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()

# Exploratory Data Analysis

We analyze:
1. Class distribution
2. Missing values
3. Comment length
4. Numeric feature relationships

WE NEED BALANCED CLASS THE LABELS ARE HIGHLY IMBALANCED

In [ ]:
#MISSING VALUES
missing = train.isnull().sum()
print(missing)
print((missing / len(train)) * 100)#MISSING VALUES CHECK
#PLOTTING OF MISSING VALUES
plt.figure(figsize=(10, 4))
missing[missing > 0].plot(kind='bar', color='salmon')
plt.title("Missing Value Counts")
plt.show()

In [ ]:
#AVERAGE COMMENT LENGTH ANALYSIS PER LABEL
train['comment_length'] = train['comment'].fillna('').apply(len)
train['word_count'] = train['comment'].fillna('').apply(lambda x: len(x.split()))
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(data=train, x='comment_length', hue='label', bins=50, kde=True)
plt.title("Comment Length by Label")
plt.xlim(0, 1000)
plt.subplot(1, 2, 2)
sns.histplot(data=train, x='word_count', hue='label', bins=50, kde=True)
plt.title("Word Count by Label")
plt.xlim(0, 200)
plt.tight_layout()
plt.show()
print("\nAverage comment length per label:")
print(train.groupby('label')['comment_length'].mean())

In [ ]:
# Numeric Feature Analysis 
numeric_features = ['emoticon_1','emoticon_2','emoticon_3',
                    'upvote','downvote','if_1','if_2',
                    'race','religion','gender','disability']
print(train[numeric_features].dtypes) #FEATURE TYPES


for col in numeric_features: #UNIQUE VALUES PER EACH FEATURE
    print(f"{col}: {sorted(train[col].dropna().unique()[:10])}")
# Encode categorical columns for correlation
corr_df = train[numeric_features + ['label']].copy().fillna(0)
# Convert object columns to numeric codes
for col in ['race', 'religion', 'gender']:
    corr_df[col] = pd.Categorical(corr_df[col]).codes
# Convert bool to int
corr_df['disability'] = corr_df['disability'].astype(int)
#PLOTTING OF DATA
plt.figure(figsize=(12, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Correlation Heatmap of Numeric Features")
plt.show()



In [ ]:
# Upvote/Downvote Distribution by Label
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='label', y='upvote', data=train)
plt.title("Upvotes by Label")
plt.ylim(0, 100)

plt.subplot(1, 2, 2)
sns.boxplot(x='label', y='downvote', data=train)
plt.title("Downvotes by Label")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

# DATA CLEANING & PREPROCESSING

In [ ]:
train = train.dropna(subset=['comment'])

cat_cols = ['race', 'religion', 'gender']
num_cols_basic = ['emoticon_1','emoticon_2','emoticon_3','upvote','downvote','if_1','if_2']

train[cat_cols] = train[cat_cols].fillna('none').astype(str)
test[cat_cols]  = test[cat_cols].fillna('none').astype(str)

for col in num_cols_basic:
    train[col] = pd.to_numeric(train[col], errors='coerce').fillna(0)
    test[col]  = pd.to_numeric(test[col], errors='coerce').fillna(0)

train['disability'] = train['disability'].astype(int)
test['disability']  = test['disability'].astype(int)

# DATETIME FEATURES
train['created_date'] = pd.to_datetime(train['created_date'], utc=True)
test['created_date']  = pd.to_datetime(test['created_date'], utc=True)

train['hour']      = train['created_date'].dt.hour
train['month']     = train['created_date'].dt.month
train['dayofweek'] = train['created_date'].dt.dayofweek

test['hour']      = test['created_date'].dt.hour
test['month']     = test['created_date'].dt.month
test['dayofweek'] = test['created_date'].dt.dayofweek

print("Missing values after cleaning:")
print(train.isnull().sum())

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s!?]', ' ', text)  # put this back
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['comment_clean'] = train['comment'].apply(clean_text)
test['comment_clean'] = test['comment'].apply(clean_text)
print("Sample cleaned comment:")
print(train['comment_clean'].iloc[0])

In [ ]:
# new features from raw comment
train['caps_count'] = train['comment'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()))
test['caps_count'] = test['comment'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()))

train['exclamation_count'] = train['comment'].apply(
    lambda x: str(x).count('!'))
test['exclamation_count'] = test['comment'].apply(
    lambda x: str(x).count('!'))

train['question_count'] = train['comment'].apply(
    lambda x: str(x).count('?'))
test['question_count'] = test['comment'].apply(
    lambda x: str(x).count('?'))
# add comment_length AND word_count to test
test['comment_length'] = test['comment'].fillna('').apply(len)
test['word_count'] = test['comment'].fillna('').apply(
    lambda x: len(str(x).split()))

print("New features added")
print(train[['caps_count','exclamation_count','question_count']].describe())

# FEATURE ENGINEERING & EXTRACTION

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# WORD LEVEL TF-IDF
tfidf_word = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=80000,
    stop_words='english',
    min_df=2,
    sublinear_tf=True
)
X_text = tfidf_word.fit_transform(train['comment_clean'])
X_test_text = tfidf_word.transform(test['comment_clean'])
print("Word TF-IDF shape:", X_text.shape)

# CHARACTER LEVEL TF-IDF
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 6),
    max_features=40000,
    min_df=2,
    sublinear_tf=True
)
X_char = tfidf_char.fit_transform(train['comment_clean'])
X_test_char = tfidf_char.transform(test['comment_clean'])
print("Char TF-IDF shape:", X_char.shape)

# NUMERIC + CATEGORICAL + DATETIME FEATURES
cat_cols = ['race', 'religion', 'gender']
num_cols = ['emoticon_1','emoticon_2','emoticon_3',
            'upvote','downvote','if_1','if_2','disability',
            'caps_count','exclamation_count','question_count',
            'comment_length','word_count',
            'hour','month','dayofweek']

train[cat_cols] = train[cat_cols].fillna('none').astype(str)
test[cat_cols]  = test[cat_cols].fillna('none').astype(str)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_cat = ohe.fit_transform(train[cat_cols])
X_test_cat = ohe.transform(test[cat_cols])

X_num = csr_matrix(train[num_cols].fillna(0).astype(float).values)
X_test_num = csr_matrix(test[num_cols].fillna(0).astype(float).values)

X_numeric = hstack([X_num, X_cat])
X_test_numeric = hstack([X_test_num, X_test_cat])
print("Numeric + Categorical feature shape:", X_numeric.shape)

X_final = hstack([X_text, X_char, X_numeric])
X_test_final = hstack([X_test_text, X_test_char, X_test_numeric])
print("Final combined feature shape:", X_final.shape)

# TRAIN-VALIDATION SPLIT

In [ ]:
y = train['label']
X_train, X_val, y_train, y_val = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

# MODEL COMPARISON (ON SMALL DATA INITIALLY)

In [ ]:
# REDUCED SUBSET AND ITERATIONS FOR SPEED
subset_size = 20000
idx = np.random.choice(X_train.shape[0], subset_size, replace=False)
X_small = X_train[idx]
y_small = y_train.iloc[idx]
results = {}

print("Model 1: Naive Bayes")
nb = MultinomialNB()
nb.fit(X_small[:, :80000], y_small)
nb_preds = nb.predict(X_val[:, :80000])
results['Naive Bayes'] = {
    'Accuracy': round(accuracy_score(y_val, nb_preds), 4),
    'Macro F1': round(f1_score(y_val, nb_preds, average='macro'), 4)
}
print("Naive Bayes →", results['Naive Bayes'])

print("Model 2: Logistic Regression")
lr = LogisticRegression(C=1, class_weight='balanced',
                        max_iter=30, solver='lbfgs', n_jobs=-1)
lr.fit(X_small, y_small)
lr_preds = lr.predict(X_val)
results['Logistic Regression'] = {
    'Accuracy': round(accuracy_score(y_val, lr_preds), 4),
    'Macro F1': round(f1_score(y_val, lr_preds, average='macro'), 4)
}
print("Logistic Regression →", results['Logistic Regression'])

print("Model 3: Linear SVM")
svm = LinearSVC(C=0.1, class_weight='balanced', max_iter=100)
svm.fit(X_small, y_small)
svm_preds = svm.predict(X_val)
results['Linear SVM'] = {
    'Accuracy': round(accuracy_score(y_val, svm_preds), 4),
    'Macro F1': round(f1_score(y_val, svm_preds, average='macro'), 4)
}
print("Linear SVM →", results['Linear SVM'])

print("Model 4: SGD")
sgd = SGDClassifier(loss='modified_huber', class_weight='balanced', random_state=42)
sgd.fit(X_small, y_small)
sgd_preds = sgd.predict(X_val)
results['SGD'] = {
    'Accuracy': round(accuracy_score(y_val, sgd_preds), 4),
    'Macro F1': round(f1_score(y_val, sgd_preds, average='macro'), 4)
}
print("SGD →", results['SGD'])

print("Model 5: LightGBM")
lgb_compare = LGBMClassifier(
    objective='multiclass', num_class=4,
    n_estimators=50,
    learning_rate=0.1,
    num_leaves=64,
    class_weight='balanced',
    n_jobs=-1, verbose=-1
)
lgb_compare.fit(X_small, y_small)
lgb_preds = lgb_compare.predict(X_val)
results['LightGBM'] = {
    'Accuracy': round(accuracy_score(y_val, lgb_preds), 4),
    'Macro F1': round(f1_score(y_val, lgb_preds, average='macro'), 4)
}
print("LightGBM →", results['LightGBM'])

# MODEL COMPARISON & ANALYSIS

In [ ]:
#MODEL COMPARISION

results_df = pd.DataFrame(results).T.sort_values('Macro F1', ascending=False)
print(results_df)

plt.figure(figsize=(12, 5)) #CONSISDERED ONLY M1SCORE AS DATA IS IMBALANCED
plt.subplot(1, 2, 1)
results_df['Macro F1'].plot(kind='bar', color='steelblue')
plt.title("Macro F1 Score Comparison")
plt.ylabel("F1 Score")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.subplot(1, 2, 2)
results_df['Accuracy'].plot(kind='bar', color='coral')
plt.title("Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Insight printed
print("\nInsight: LightGBM and LR typically perform best on text classification.")
print("SVM is fast and competitive. NB is weakest but fastest baseline.")
print("Ensemble of top models gives best overall score.")

# HYPERPARAMETER TUNNING

In [ ]:
# HYPERPARAMETER TUNING
print("Tuning LightGBM parameters")
param_combinations = [
    {'n_estimators': 100, 'learning_rate': 0.1,  'num_leaves': 64},
    {'n_estimators': 200, 'learning_rate': 0.05, 'num_leaves': 128},
    {'n_estimators': 300, 'learning_rate': 0.03, 'num_leaves': 128},
]
best_f1, best_params = 0, None
for params in param_combinations:
    m = LGBMClassifier(
        objective='multiclass', num_class=4,
        class_weight='balanced',
        n_jobs=-1, verbose=-1,
        **params
    )
    m.fit(X_small, y_small)
    preds = m.predict(X_val)
    f1 = f1_score(y_val, preds, average='macro')
    print(f"  params={params} → Macro F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_params = params
print(f"\nBest params: {best_params}")
print(f"Best Macro F1: {best_f1:.4f}")


# FINAL SUBMISSION

In [ ]:
print("Training LightGBM...")
lgb_final = LGBMClassifier(
    objective='multiclass', num_class=4,
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=128,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight='balanced',
    n_jobs=-1, verbose=-1,
    random_state=42
)
lgb_final.fit(X_final, y)
lgb_probs = lgb_final.predict_proba(X_test_final)
print("LightGBM done.")

final_preds = np.argmax(lgb_probs, axis=1)
sample_sub["label"] = final_preds
sample_sub.to_csv("submission.csv", index=False)
print("submission completed")
print("Distribution:", pd.Series(final_preds).value_counts().to_dict())